# Level 14 — Risk Budgeting and Equal Risk Contribution

**Audience:** portfolio analysts who want to allocate portfolio risk rather
than capital alone.

**Prerequisites:** Levels 2 and 12, covariance matrices, and portfolio
volatility.

**Learning goals**

1. distinguish capital weights from normalized risk contributions;
2. construct equal-risk-contribution (ERC) weights;
3. target a non-equal risk budget;
4. create transparent equal- and capitalization-weight policies.

The examples use a synthetic covariance matrix and market capitalizations.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.portfolio import (
    capitalization_weights,
    capped_equal_weights,
    equal_risk_contribution_weights,
    equal_weights,
    risk_contributions,
    target_risk_contribution_weights,
)

In [ ]:
assets = pd.Index(["Equity", "Bonds", "Real assets"])
covariance = pd.DataFrame(
    [
        [0.0400, 0.0030, 0.0090],
        [0.0030, 0.0100, 0.0020],
        [0.0090, 0.0020, 0.0250],
    ],
    index=assets,
    columns=assets,
)
covariance

## 2. Equal capital is not equal risk

Equal weights allocate one-third of capital to every asset. Different
volatilities and correlations mean their risk contributions need not be equal.

In [ ]:
equal_capital = equal_weights(assets)
pd.DataFrame(
    {
        "capital_weight": equal_capital,
        "risk_contribution": risk_contributions(
            equal_capital,
            covariance,
        ),
    }
)

## 3. Equal-risk-contribution portfolio

ERC solves for long-only, fully-invested weights whose normalized
contributions to portfolio volatility are equal.

In [ ]:
erc_weights = equal_risk_contribution_weights(covariance)
pd.DataFrame(
    {
        "capital_weight": erc_weights,
        "risk_contribution": risk_contributions(
            erc_weights,
            covariance,
        ),
    }
)

## 4. Target a deliberate risk budget

The target below assigns 50% of total ex-ante risk to Equity, 20% to Bonds,
and 30% to Real assets. Risk budgets must be strictly positive and sum to one.

In [ ]:
target_budget = pd.Series(
    [0.50, 0.20, 0.30],
    index=assets,
    name="target_contribution",
)
target_weights = target_risk_contribution_weights(
    target_budget,
    covariance,
)
pd.DataFrame(
    {
        "target_risk": target_budget,
        "achieved_risk": risk_contributions(
            target_weights,
            covariance,
        ),
        "capital_weight": target_weights,
    }
)

## 5. Transparent weighting policies

Capitalization weights use caller-supplied market values. Capped equal weights
can screen very small assets and prevent an equal-weight rule from exceeding a
multiple of capitalization weight.

In [ ]:
market_caps = pd.Series(
    [700.0, 200.0, 100.0],
    index=assets,
    name="market_capitalization",
)
pd.concat(
    {
        "Equal": equal_weights(assets),
        "Capitalization": capitalization_weights(market_caps),
        "Screened and capped": capped_equal_weights(
            market_caps,
            minimum_capitalization_weight=0.05,
            maximum_multiple_of_cap_weight=2.0,
        ),
    },
    axis=1,
)

## Exercise — tilt the risk budget

Change the target from `[0.50, 0.20, 0.30]` to `[0.30, 0.40, 0.30]`. Which
capital weight changes most? Verify the achieved contributions directly.

In [ ]:
# Try it here.
defensive_budget = pd.Series(
    [0.30, 0.40, 0.30],
    index=assets,
)

### Answer scaffold

In [ ]:
defensive_weights = target_risk_contribution_weights(
    defensive_budget,
    covariance,
)
pd.DataFrame(
    {
        "original_weight": target_weights,
        "defensive_weight": defensive_weights,
        "weight_change": defensive_weights - target_weights,
        "achieved_risk": risk_contributions(
            defensive_weights,
            covariance,
        ),
    }
)

## Interpretation and pitfalls

- Risk budgets depend entirely on the covariance estimate.
- Equal risk contribution is not equal capital weight.
- Ex-ante contributions can differ from realized contributions.
- Long-only target budgets may be infeasible or numerically fragile for some
  covariance structures.
- Static weighting policies do not define rebalance timing. Feed dated target
  weights into `run_weight_backtest` to model drift, turnover, and costs.

Next: estimate fund exposures to labelled return factors.